In [1]:
import warnings
warnings.filterwarnings("ignore", message="DataFrame.sql_ctx is an internal property")


## Spark Session

In [20]:
from pyspark.sql import SparkSession
from graphframes import GraphFrame
from pyspark.sql.functions import (
    array_sort, array, 
    explode, lit, least, 
    col, desc, count, 
    collect_list,
    sum as Fsum
)
from pyspark.sql.window import Window


# Create Spark session (configured with DeltaCatalog in Docker)
spark = SparkSession.builder \
    .appName("FlightGraphAnalysis") \
    .getOrCreate()

# Optional: for cleaner notebook output
spark.conf.set("spark.sql.shuffle.partitions", spark._sc.defaultParallelism)
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 500)

print("Spark session is ready")


Spark session is ready


## Schema and dataframe

In [4]:
file_path = "/home/jovyan/input/2009.csv"

df = spark.read.csv(file_path, header=True, inferSchema=True)
df.printSchema()
display(df)

root
 |-- FL_DATE: date (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |-- WHEELS_OFF: double (nullable = true)
 |-- WHEELS_ON: double (nullable = true)
 |-- TAXI_IN: double (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_TIME: double (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- CRS_ELAPSED_TIME: double (nullable = true)
 |-- ACTUAL_ELAPSED_TIME: double (nullable = true)
 |-- AIR_TIME: double (nullable = true)
 |-- DISTANCE: double (nullable = true)
 |-- CARRIER_DELAY: double (nullable = true)
 |-- WEATHER_DELAY: doub

FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,Unnamed: 27
2009-01-01,XE,1204,DCA,EWR,1100,1058.0,-2.0,18.0,1116.0,1158.0,8.0,1202,1206.0,4.0,0.0,NULL,0.0,62.0,68.0,42.0,199.0,NULL,NULL,NULL,NULL,NULL,NULL
2009-01-01,XE,1206,EWR,IAD,1510,1509.0,-1.0,28.0,1537.0,1620.0,4.0,1632,1624.0,-8.0,0.0,NULL,0.0,82.0,75.0,43.0,213.0,NULL,NULL,NULL,NULL,NULL,NULL
2009-01-01,XE,1207,EWR,DCA,1100,1059.0,-1.0,20.0,1119.0,1155.0,6.0,1210,1201.0,-9.0,0.0,NULL,0.0,70.0,62.0,36.0,199.0,NULL,NULL,NULL,NULL,NULL,NULL
2009-01-01,XE,1208,DCA,EWR,1240,1249.0,9.0,10.0,1259.0,1336.0,9.0,1357,1345.0,-12.0,0.0,NULL,0.0,77.0,56.0,37.0,199.0,NULL,NULL,NULL,NULL,NULL,NULL
2009-01-01,XE,1209,IAD,EWR,1715,1705.0,-10.0,24.0,1729.0,1809.0,13.0,1900,1822.0,-38.0,0.0,NULL,0.0,105.0,77.0,40.0,213.0,NULL,NULL,NULL,NULL,NULL,NULL
2009-01-01,XE,1212,ATL,EWR,1915,1913.0,-2.0,19.0,1932.0,2108.0,15.0,2142,2123.0,-19.0,0.0,NULL,0.0,147.0,130.0,96.0,745.0,NULL,NULL,NULL,NULL,NULL,NULL
2009-01-01,XE,1212,CLE,ATL,1645,1637.0,-8.0,12.0,1649.0,1820.0,5.0,1842,1825.0,-17.0,0.0,NULL,0.0,117.0,108.0,91.0,554.0,NULL,NULL,NULL,NULL,NULL,NULL
2009-01-01,XE,1214,DCA,EWR,1915,1908.0,-7.0,9.0,1917.0,1953.0,34.0,2035,2027.0,-8.0,0.0,NULL,0.0,80.0,79.0,36.0,199.0,NULL,NULL,NULL,NULL,NULL,NULL
2009-01-01,XE,1215,EWR,DCA,1715,1710.0,-5.0,28.0,1738.0,1819.0,4.0,1838,1823.0,-15.0,0.0,NULL,0.0,83.0,73.0,41.0,199.0,NULL,NULL,NULL,NULL,NULL,NULL
2009-01-01,XE,1217,EWR,DCA,1300,1255.0,-5.0,15.0,1310.0,1349.0,7.0,1408,1356.0,-12.0,0.0,NULL,0.0,68.0,61.0,39.0,199.0,NULL,NULL,NULL,NULL,NULL,NULL


#### Keep only necessary columns

In [5]:
flight_df = df.select("ORIGIN", "DEST")

## Overall Statistics

In [6]:
import pyspark.sql.functions as F
from graphframes import GraphFrame

# Define vertices: all unique airport codes
flight_vertices = (
    flight_df.select(F.col("Origin").alias("id"))
      .union(df.select(F.col("Dest").alias("id")))
      .distinct()
)

# Define edges: flight routes from Origin to Dest
flight_edges = (
    flight_df.select(F.col("Origin").alias("src"), F.col("Dest").alias("dst"))
)

# Create the GraphFrame
flight_graph = GraphFrame(flight_vertices, flight_edges)

# Print number of vertices and edges
print("Number of vertices (airports):", flight_graph.vertices.count())
print("Number of edges (flights):", flight_graph.edges.count())

Number of vertices (airports): 296
Number of edges (flights): 6429338


## Query 1 – Degree and Triangle Statistics


We analyze the flight graph by computing:
- **In-degree**: number of incoming flights to each airport.
- **Out-degree**: number of outgoing flights from each airport.
- **Total degree**: sum of in- and out-degrees.
- **Triangle count**: number of closed triplets A → B → C → A (computed using native Spark SQL joins).


In [14]:
# Count outgoing flights (from an airport)
out_deg = flight_graph.outDegrees.withColumnRenamed("id", "airport").fillna(0)

# Count incoming flights (to an airport)
in_deg = flight_graph.inDegrees.withColumnRenamed("id", "airport").fillna(0)

# If needed to calculate manually
# out_deg = flight_edges.groupBy("src").agg(count("*").alias("outDegree"))
# in_deg = flight_edges.groupBy("dst").agg(count("*").alias("inDegree"))
# degree_df = in_deg.join(out_deg, in_deg["dst"] == out_deg["src"], "outer")
#                   .fillna(0)
# degree_df = degree_df.withColumn("total_degree", col("inDegree") + col("outDegree"))

# Join and calculate total degree
degree_df = flight_graph.degrees.withColumnRenamed("id", "airport").withColumnRenamed("degree", "total_degree").fillna(0)

degree_df = degree_df.join(in_deg, on="airport", how="outer") \
                     .join(out_deg, on="airport", how="outer")
degree_df.orderBy(desc("total_degree")).show(10)

+-------+------------+--------+---------+
|airport|total_degree|inDegree|outDegree|
+-------+------------+--------+---------+
|    ATL|      834906|  417457|   417449|
|    ORD|      627617|  313769|   313848|
|    DFW|      528794|  264398|   264396|
|    DEN|      471375|  235700|   235675|
|    LAX|      385795|  192916|   192879|
|    PHX|      366993|  183491|   183502|
|    IAH|      364185|  182088|   182097|
|    LAS|      307977|  153984|   153993|
|    DTW|      304156|  152075|   152081|
|    SFO|      273020|  136532|   136488|
+-------+------------+--------+---------+
only showing top 10 rows



In [16]:
flight_graph.triplets.show(10)

+-----+----------+-----+
|  src|      edge|  dst|
+-----+----------+-----+
|{JAX}|{JAX, CLE}|{CLE}|
|{JAX}|{JAX, IAH}|{IAH}|
|{MEM}|{MEM, IAH}|{IAH}|
|{MEM}|{MEM, IAH}|{IAH}|
|{BTV}|{BTV, CLE}|{CLE}|
|{BOS}|{BOS, CLE}|{CLE}|
|{MEM}|{MEM, IAH}|{IAH}|
|{MEM}|{MEM, IAH}|{IAH}|
|{JAX}|{JAX, IAH}|{IAH}|
|{JAX}|{JAX, IAH}|{IAH}|
+-----+----------+-----+
only showing top 10 rows



### Count edges (to avoid duplicate operations)

In [19]:
edge_counts = flight_graph.triplets \
    .groupBy(col("src.id").alias("src"), col("dst.id").alias("dst")) \
    .agg(count("*").alias("count"))
edge_counts.orderBy(col("count").desc()).show(10)

+---+---+-----+
|src|dst|count|
+---+---+-----+
|OGG|HNL|13330|
|HNL|OGG|12969|
|SFO|LAX|12134|
|LAX|SFO|12127|
|SAN|LAX|11020|
|LAX|SAN|11017|
|BOS|LGA|10874|
|LGA|BOS|10870|
|LAX|LAS|10683|
|LAS|LAX|10662|
+---+---+-----+
only showing top 10 rows



#### Now, find how many pairs of edges with a common vertice there are, that are not backwards trips (dst1 == src2 && dst2 != src1). To get the count, multiply the edge pair counts (for each edge pair x, y, with out deg dx and dy, for each edge in x, there are dy possibilities => dx*dy total possibilitiees)

In [26]:
forward_pairs = edge_counts.alias("e1").join(
    edge_counts.alias("e2"),
    col("e1.dst") == col("e2.src")
).filter(
    col("e1.src") != col("e2.dst")
).select("e1.src", "e2.dst", "e1.count", "e2.count")
forward_pairs.show(10)

+---+---+-----+-----+
|src|dst|count|count|
+---+---+-----+-----+
|ATL|CLE| 5614|  240|
|DTW|CLE|  695|  240|
|LAS|CLE|  364|  240|
|BHM|CLE|  362|  240|
|LGA|CLE| 1174|  240|
|CLT|CLE| 2370|  240|
|CVG|CLE|  562|  240|
|TPA|CLE|  975|  240|
|BWI|CLE| 1065|  240|
|PHL|CLE| 1647|  240|
+---+---+-----+-----+
only showing top 10 rows



In [27]:
forward_pairs = forward_pairs.withColumn(
    "total_count",
    col("e1.count")*col("e2.count")
).select("src", "dst", "total_count").withColumnRenamed("total_count", "count")
forward_pairs.show(10)

+---+---+-------+
|src|dst|  count|
+---+---+-------+
|ATL|CLE|1347360|
|DTW|CLE| 166800|
|LAS|CLE|  87360|
|BHM|CLE|  86880|
|LGA|CLE| 281760|
|CLT|CLE| 568800|
|CVG|CLE| 134880|
|TPA|CLE| 234000|
|BWI|CLE| 255600|
|PHL|CLE| 395280|
+---+---+-------+
only showing top 10 rows



#### Groupby src and dst and sum count

In [28]:
forward_pairs = forward_pairs.groupBy(
    col("src"),
    col("dst")
).agg(
    Fsum("count").alias("count")
)
forward_pairs.show(10)

+---+---+---------+
|src|dst|    count|
+---+---+---------+
|LAS|CLE|122309639|
|HOU|CLE| 36243084|
|LAS|IAH|351433317|
|BWI|IAH|210371851|
|DAL|IAH| 91820831|
|CAE|IAH| 37443095|
|OMA|IAH| 83153797|
|SAV|IAH| 44580302|
|MLI|IAH| 36495227|
|FSM|IAH| 10839689|
+---+---+---------+
only showing top 10 rows



#### Finally, multiply all src -> dst in forward_pairs by dst -> src in edge_counts (complete the circle)

In [30]:
triangle_counts = forward_pairs.alias("p").join(
    edge_counts.alias("e"),
    (col("p.src") == col("e.dst")) & (col("e.src") == col("p.dst"))
).withColumn(
    "total_count",
    col("e.count")*col("p.count")
).select("p.src", "total_count").withColumnRenamed("total_count", "count").withColumnRenamed("src", "airport")
triangle_counts.orderBy(col("count").desc()).show(10)

+-------+-------------+
|airport|        count|
+-------+-------------+
|    ATL|8504093691324|
|    DFW|8478731863227|
|    ORD|7971636381240|
|    ATL|7801405018632|
|    LAS|6538795878942|
|    LAX|6510419180928|
|    LAX|6464248352300|
|    SFO|6425748260686|
|    PHX|5945603323328|
|    DEN|5931468741096|
+-------+-------------+
only showing top 10 rows



### Query 2: Global Triangle Count

In this query, we compute the **total number of unique triangles**.

A triangle in the graph is defined as a cycle of three connected airports:  
`A → B → C → A`

#### Approach:
1. Keep only unique edges
2. For each edge e1, count all the combinations of edges e2 and e3, where (e1.src == e3.dst) & (e2.src == e1.dst) & (e3.src == e2.dst)
3. No need to check for (e2.src != e3.dst), since if (e2.src == e3.dst) => (e1.src == e2.src) => (e1.src == e1.dst) (we can add a check just in case, that e1.src != e1.dst).
4. Remove duplicates (A, B, C) and (C, A, B)

In [36]:
unique_edges = flight_graph.triplets \
    .select(col("src.id").alias("src"), col("dst.id").alias("dst")) \
    .distinct() \
    .filter(col("src") != col("dst")) \
    .cache()

e1 = unique_edges.alias("e1")
e2 = unique_edges.alias("e2")
e3 = unique_edges.alias("e3")

triangles = (e1
             .join(e2, col("e1.dst") == col("e2.src"))
             .join(e3, (col("e2.dst") == col("e3.src")) & (col("e3.dst") == col("e1.src")))
              # Create sorted array from all vertices.
              # Needed, since [A,B,C] == [C,A,B], and we want to find these duplicates
              .select(array_sort(array("e1.src", "e2.src", "e3.src")).alias("triangle"))
              # Remove duplicates
              .distinct()
)
triangle_count = triangles.count()
triangle_count

16014

### Query 3: Degree Centrality Using GraphFrames (Native Spark Logic)

In this query, we compute the **degree centrality** for each airport.  
Degree centrality measures how connected a node (airport) is in the graph.

We used the GraphFrame structure (`flight_graph.vertices`, `flight_graph.edges`).

#### Formula:

$$
\text{Degree Centrality} = \frac{\text{Total Degree}}{N - 1}
$$

Where:

- **Total Degree** = in-degree + out-degree  
- **\( N \)** = number of unique airports (nodes in the graph)

This gives a score between **0 and 1** for each airport, indicating how central it is in the network.


In [37]:
# Use GraphFrame structure
g_edges = flight_graph.edges
g_vertices = flight_graph.vertices

# Compute in-degree and out-degree
in_deg = g_edges.groupBy("dst").count().withColumnRenamed("dst", "id").withColumnRenamed("count", "in_degree")
out_deg = g_edges.groupBy("src").count().withColumnRenamed("src", "id").withColumnRenamed("count", "out_degree")

# Join and compute total degree
deg_centrality = in_deg.join(out_deg, on="id", how="outer").fillna(0)
deg_centrality = deg_centrality.withColumn("total_degree", col("in_degree") + col("out_degree"))

# Normalize by total number of airports
num_airports = g_vertices.count()
deg_centrality = deg_centrality.withColumn(
    "degree_centrality", col("total_degree") / (num_airports - 1)
)

# Show top 10 most central airports
deg_centrality.orderBy(desc("degree_centrality")).show(10)


+---+---------+----------+------------+------------------+
| id|in_degree|out_degree|total_degree| degree_centrality|
+---+---------+----------+------------+------------------+
|ATL|   417457|    417449|      834906|2830.1898305084746|
|ORD|   313769|    313848|      627617| 2127.515254237288|
|DFW|   264398|    264396|      528794|1792.5220338983052|
|DEN|   235700|    235675|      471375|1597.8813559322034|
|LAX|   192916|    192879|      385795|1307.7796610169491|
|PHX|   183491|    183502|      366993|  1244.04406779661|
|IAH|   182088|    182097|      364185|1234.5254237288136|
|LAS|   153984|    153993|      307977|1043.9898305084746|
|DTW|   152075|    152081|      304156|1031.0372881355931|
|SFO|   136532|    136488|      273020| 925.4915254237288|
+---+---------+----------+------------+------------------+
only showing top 10 rows



#### Check Degree Centrality using GraphFrames (built-in)

In [38]:
# Use built-in total degree (in + out)
gf_degrees = flight_graph.degrees

# Total number of airports
num_airports = flight_graph.vertices.count()

# Calculate degree centrality
gf_deg_centrality = gf_degrees.withColumn(
    "degree_centrality", col("degree") / (num_airports - 1)
)

# Show top 10 airports by centrality
gf_deg_centrality.orderBy(desc("degree_centrality")).show(10)


+---+------+------------------+
| id|degree| degree_centrality|
+---+------+------------------+
|ATL|834906|2830.1898305084746|
|ORD|627617| 2127.515254237288|
|DFW|528794|1792.5220338983052|
|DEN|471375|1597.8813559322034|
|LAX|385795|1307.7796610169491|
|PHX|366993|  1244.04406779661|
|IAH|364185|1234.5254237288136|
|LAS|307977|1043.9898305084746|
|DTW|304156|1031.0372881355931|
|SFO|273020| 925.4915254237288|
+---+------+------------------+
only showing top 10 rows



### Query 4: PageRank (Natively on Spark using GraphFrames Structure)

In this query, we implement the **PageRank algorithm manually using native Spark DataFrame operations**.

**PageRank** is an iterative algorithm that measures the importance of each node (airport) based on the structure of the graph. An airport gets a higher rank if it is linked to by other high-ranking airports.

#### Formula:
Each node's score is updated as:

$$
\text{PR}(A) = \frac{1 - d}{N} + d \cdot \sum_{i \in \text{in-neighbors of A}} \frac{\text{PR}(i)}{\text{out-degree}(i)}
$$

Where:
- \( d \) = damping factor (commonly set to 0.85)
- \( N \) = total number of nodes
- Iteration continues until convergence (we fix a number of steps)


In [ ]:
# Parameters
damping_factor = 0.85
num_nodes = flight_graph.vertices.count()
num_iterations = 10

# Initialize ranks equally
ranks = flight_graph.vertices.withColumn("rank", lit(1.0 / num_nodes)).cache()

# Get edges (links)
links = flight_graph.edges.groupBy("src").agg(F.collect_list("dst").alias("neighbors"))

# Iteratively compute PageRank
for i in range(num_iterations):
    # Join ranks with links
    contribs = ranks.join(links, ranks.id == links.src, "left_outer") \
        .select("id", "rank", "neighbors") \
        .rdd.flatMap(lambda row: 
            [(nbr, row.rank / len(row.neighbors)) for nbr in row.neighbors] if row.neighbors else []
        ).toDF(["id", "contribution"])

    # Aggregate contributions
    ranks = contribs.groupBy("id").agg(F.sum("contribution").alias("total_contrib")).cache()

    # Recalculate ranks
    ranks = ranks.withColumn(
        "rank", (lit(1 - damping_factor) / num_nodes) + damping_factor * col("total_contrib")
    ).select("id", "rank")

# Show top 10 airports by PageRank
ranks.orderBy(col("rank").desc()).show(10)


#### PageRank with graphframes functions

In [ ]:
pagerank_check = flight_graph.pageRank(resetProbability=0.15, maxIter=10)
pagerank_check.vertices.select("id", "pagerank").orderBy(desc("pagerank")).show(10)


### Query 5: Finding the Group of Most Connected Airports

In this query, we aim to find **the group of airports that are most interconnected**, i.e., a cluster where airports have high mutual connectivity.

We interpret this as finding the **largest connected component** in the graph — a set of airports that are all reachable from each other, either directly or through other airports.

#### Key idea:
Use a **label propagation-like approach** to assign each node a component ID and find the largest group by size.

In [ ]:
# Initialize each airport's component ID to itself
components = flight_graph.vertices.withColumn("component", col("id")).cache()

edges = flight_graph.edges.select("src", "dst")

# Propagate minimum ID across edges iteratively
for i in range(10):  # number of iterations can be adjusted
    joined = edges.join(components, edges.src == components.id) \
                  .select(edges.dst.alias("id"), components.component)
    
    min_components = components.join(joined, "id", "outer") \
        .withColumn("component", least(col("component"), col("component").over(Window.partitionBy("id")))) \
        .select("id", "component").distinct()

    components = min_components.cache()

# Count airports per component and show the largest group
most_connected_group = components.groupBy("component") \
                                 .count() \
                                 .orderBy(col("count").desc())

most_connected_group.show(10)


#### Same with graphframes functions

In [ ]:
connected_components = flight_graph.connectedComponents()
connected_components.groupBy("component").count().orderBy(desc("count")).show(10)
